# Scipy Sparse Matrices

**Sparse Matrices** are very nice in some situations.  

For example, in some machine learning tasks, especially those associated
with textual analysis, the data may be mostly zeros.  

Storing all these zeros is very inefficient.  

We can create and manipulate sparse matrices as follows:

In [1]:
import numpy as np

In [2]:
# Create a random array with a lot of zeros
X = np.random.random((10, 5))
print(X)

[[0.11869285 0.01160985 0.50023375 0.51604401 0.19930497]
 [0.13602766 0.22830205 0.08284765 0.46881382 0.35847626]
 [0.53970514 0.75779087 0.24097231 0.76510523 0.8905599 ]
 [0.10733453 0.34337304 0.45607971 0.12214688 0.56442636]
 [0.50672529 0.4893213  0.53403666 0.03636269 0.93620906]
 [0.03497334 0.3554594  0.11274914 0.13840021 0.73471743]
 [0.28836342 0.66080851 0.585271   0.33561443 0.89308652]
 [0.84379968 0.75173721 0.3769161  0.12958055 0.80468254]
 [0.03576674 0.91215374 0.46996007 0.50944584 0.24677459]
 [0.30296494 0.60797252 0.21413743 0.84752349 0.69742071]]


In [3]:
X[X < 0.7] = 0
print(X)

[[0.         0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.        ]
 [0.         0.75779087 0.         0.76510523 0.8905599 ]
 [0.         0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.93620906]
 [0.         0.         0.         0.         0.73471743]
 [0.         0.         0.         0.         0.89308652]
 [0.84379968 0.75173721 0.         0.         0.80468254]
 [0.         0.91215374 0.         0.         0.        ]
 [0.         0.         0.         0.84752349 0.        ]]


In [4]:
from scipy import sparse

# turn X into a csr (Compressed-Sparse-Row) matrix
X_csr = sparse.csr_matrix(X)
print(X_csr)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 11 stored elements and shape (10, 5)>
  Coords	Values
  (2, 1)	0.7577908654823686
  (2, 3)	0.7651052345258225
  (2, 4)	0.8905599030439532
  (4, 4)	0.9362090634213527
  (5, 4)	0.7347174262685747
  (6, 4)	0.8930865193006828
  (7, 0)	0.8437996806964837
  (7, 1)	0.7517372117340332
  (7, 4)	0.8046825438138948
  (8, 1)	0.9121537394322848
  (9, 3)	0.8475234867100796


In [5]:
# convert the sparse matrix to a dense array
print(X_csr.toarray())

[[0.         0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.        ]
 [0.         0.75779087 0.         0.76510523 0.8905599 ]
 [0.         0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.93620906]
 [0.         0.         0.         0.         0.73471743]
 [0.         0.         0.         0.         0.89308652]
 [0.84379968 0.75173721 0.         0.         0.80468254]
 [0.         0.91215374 0.         0.         0.        ]
 [0.         0.         0.         0.84752349 0.        ]]


In [6]:
# Sparse matrices support linear algebra:
y = np.random.random(X_csr.shape[1])
z1 = X_csr.dot(y)
z2 = X.dot(y)
np.allclose(z1, z2)

True

* The CSR representation can be very efficient for computations, but it is not as good for adding elements.  

* For that, the **LIL** (List-In-List) representation is better:

In [7]:
# Create an empty LIL matrix and add some items
X_lil = sparse.lil_matrix((5, 5))

for i, j in np.random.randint(0, 5, (15, 2)):
    X_lil[i, j] = i + j

print(X_lil)
print(X_lil.toarray())

<List of Lists sparse matrix of dtype 'float64'
	with 10 stored elements and shape (5, 5)>
  Coords	Values
  (0, 1)	1.0
  (0, 4)	4.0
  (1, 1)	2.0
  (1, 3)	4.0
  (2, 3)	5.0
  (3, 0)	3.0
  (3, 1)	4.0
  (3, 4)	7.0
  (4, 3)	7.0
  (4, 4)	8.0
[[0. 1. 0. 0. 4.]
 [0. 2. 0. 4. 0.]
 [0. 0. 0. 5. 0.]
 [3. 4. 0. 0. 7.]
 [0. 0. 0. 7. 8.]]


* Often, once an LIL matrix is created, it is useful to convert it to a CSR format 
    * **Note**: many scikit-learn algorithms require CSR or CSC format

In [8]:
X_csr = X_lil.tocsr()
print(X_csr)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 10 stored elements and shape (5, 5)>
  Coords	Values
  (0, 1)	1.0
  (0, 4)	4.0
  (1, 1)	2.0
  (1, 3)	4.0
  (2, 3)	5.0
  (3, 0)	3.0
  (3, 1)	4.0
  (3, 4)	7.0
  (4, 3)	7.0
  (4, 4)	8.0


There are several other sparse formats that can be useful for various problems:

- `CSC` (compressed sparse column)
- `BSR` (block sparse row)
- `COO` (coordinate)
- `DIA` (diagonal)
- `DOK` (dictionary of keys)

## CSC - Compressed Sparse Column

**Advantages of the CSC format**

    * efficient arithmetic operations CSC + CSC, CSC * CSC, etc.
    * efficient column slicing
    * fast matrix vector products (CSR, BSR may be faster)

**Disadvantages of the CSC format**

    * slow row slicing operations (consider CSR)
    * changes to the sparsity structure are expensive (consider LIL or DOK)

### BSR - Block Sparse Row

The Block Compressed Row (`BSR`) format is very similar to the Compressed Sparse Row (`CSR`) format. 

BSR is appropriate for sparse matrices with *dense sub matrices* like the example below. 

Block matrices often arise in *vector-valued* finite element discretizations. 

In such cases, BSR is **considerably more efficient** than CSR and CSC for many sparse arithmetic operations.

In [9]:
from scipy.sparse import bsr_matrix

indptr = np.array([0, 2, 3, 6])
indices = np.array([0, 2, 2, 0, 1, 2])
data = np.array([1, 2, 3, 4, 5, 6]).repeat(4).reshape(6, 2, 2)
bsr_matrix((data,indices,indptr), shape=(6, 6)).toarray()

array([[1, 1, 0, 0, 2, 2],
       [1, 1, 0, 0, 2, 2],
       [0, 0, 0, 0, 3, 3],
       [0, 0, 0, 0, 3, 3],
       [4, 4, 5, 5, 6, 6],
       [4, 4, 5, 5, 6, 6]])

## COO - Coordinate Sparse Matrix

**Advantages of the CSC format**

    * facilitates fast conversion among sparse formats
    * permits duplicate entries (see example)
    * very fast conversion to and from CSR/CSC formats

**Disadvantages of the CSC format**

    * does not directly support arithmetic operations and slicing
    
** Intended Usage**

    * COO is a fast format for constructing sparse matrices
    * Once a matrix has been constructed, convert to CSR or CSC format for fast arithmetic and matrix vector
    operations
    * By default when converting to CSR or CSC format, duplicate (i,j) entries will be summed together. 
    This facilitates efficient construction of finite element matrices and the like.


## DOK - Dictionary of Keys

Sparse matrices can be used in arithmetic operations: they support addition, subtraction, multiplication, division, and matrix power.

Allows for efficient O(1) access of individual elements. Duplicates are not allowed. Can be efficiently converted to a coo_matrix once constructed.

In [10]:
from scipy.sparse import dok_matrix
S = dok_matrix((5, 5), dtype=np.float32)
for i in range(5):
    for j in range(i, 5):
        S[i,j] = i+j
        
S.toarray()

array([[0., 1., 2., 3., 4.],
       [0., 2., 3., 4., 5.],
       [0., 0., 4., 5., 6.],
       [0., 0., 0., 6., 7.],
       [0., 0., 0., 0., 8.]], dtype=float32)

The ``scipy.sparse`` submodule also has a lot of functions for sparse matrices
including linear algebra, sparse solvers, graph algorithms, and much more.

# Exercises

## Ex 1.1

Create a big numpy **dense** matrix filled with random numbers in 
`[0, 1)`.
Generate a random number within this range and subsitute all the elements in the matrix **less than** this number with a zero.

Save resulting matrix as a `DOK` sparse matrix

In [11]:
# create a big dense matrix with random numbers in [0, 1)
big_matrix = np.random.random((20, 20))

# generate a random threshold within the same range
threshold = np.random.random()
print('threshold:', threshold)

# substitute all elements less than the threshold with zero
big_matrix[big_matrix < threshold] = 0

# save the result as a DOK sparse matrix
big_matrix_dok = sparse.dok_matrix(big_matrix)
print(big_matrix_dok)

threshold: 0.5005157968862048
<Dictionary Of Keys sparse matrix of dtype 'float64'
	with 205 stored elements and shape (20, 20)>
  Coords	Values
  (0, 0)	0.5521962672639834
  (0, 3)	0.5816470451185849
  (0, 5)	0.9243219302866127
  (0, 6)	0.7043710253359889
  (0, 7)	0.7278241508229609
  (0, 9)	0.826095373801721
  (0, 10)	0.7793792802881742
  (0, 13)	0.9859829099072973
  (0, 14)	0.7211925020798158
  (0, 15)	0.9262272273855527
  (0, 18)	0.7813198668454236
  (1, 2)	0.5224096008867068
  (1, 3)	0.9378208744783684
  (1, 4)	0.5111848579946661
  (1, 6)	0.6986067383848327
  (1, 7)	0.5564012107086548
  (1, 8)	0.7243469761947204
  (1, 9)	0.6124158279493453
  (1, 11)	0.5421578480868815
  (1, 12)	0.7582696458549328
  (1, 13)	0.5291737332843923
  (1, 14)	0.6753479045885968
  (1, 15)	0.6894361774099941
  (1, 17)	0.8418122506641498
  (2, 0)	0.5193555344292987
  :	:
  (17, 3)	0.7148140597806555
  (17, 8)	0.8568075141261509
  (17, 9)	0.7044645287186687
  (17, 13)	0.7591583976010616
  (17, 14)	0.639740261

## Ex 1.2

Repeat the previous exercise, but this time use a `CSR` sparse matrix.

In [12]:
# repeat the previous exercise, this time using a CSR sparse matrix
big_matrix2 = np.random.random((20, 20))

threshold2 = np.random.random()
print('threshold:', threshold2)

big_matrix2[big_matrix2 < threshold2] = 0

big_matrix_csr = sparse.csr_matrix(big_matrix2)
print(big_matrix_csr)

threshold: 0.10516504134176219
<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 353 stored elements and shape (20, 20)>
  Coords	Values
  (0, 0)	0.2506811747389758
  (0, 1)	0.6848774965502148
  (0, 2)	0.738434486663643
  (0, 3)	0.42975586531185794
  (0, 4)	0.7335350409399324
  (0, 6)	0.7615793712807801
  (0, 7)	0.7545606424255715
  (0, 8)	0.1061700862482805
  (0, 9)	0.19300219271904406
  (0, 10)	0.7630640791044943
  (0, 11)	0.552363262495868
  (0, 12)	0.997385839515126
  (0, 13)	0.22446716391582433
  (0, 14)	0.9630380729265029
  (0, 15)	0.8207869092258588
  (0, 16)	0.10907549131432426
  (0, 17)	0.5302406300809884
  (0, 18)	0.9392626593027495
  (0, 19)	0.7468391104030899
  (1, 0)	0.34272400526704205
  (1, 1)	0.9969930068806062
  (1, 2)	0.767825064387232
  (1, 3)	0.48239215080554554
  (1, 4)	0.17243950915374695
  (1, 5)	0.3340768959191478
  :	:
  (18, 10)	0.8678574247725769
  (18, 11)	0.40655835501187
  (18, 12)	0.5938444243693917
  (18, 13)	0.6812725061013015
  (18, 14)	0.5

## Ex 1.3

Transform the previously generated sparse matrix back to a full dense `numpy.array`.

In [13]:
# transform the sparse matrix back into a full dense numpy array
dense_again = big_matrix_csr.toarray()
dense_again

array([[0.25068117, 0.6848775 , 0.73843449, 0.42975587, 0.73353504,
        0.        , 0.76157937, 0.75456064, 0.10617009, 0.19300219,
        0.76306408, 0.55236326, 0.99738584, 0.22446716, 0.96303807,
        0.82078691, 0.10907549, 0.53024063, 0.93926266, 0.74683911],
       [0.34272401, 0.99699301, 0.76782506, 0.48239215, 0.17243951,
        0.3340769 , 0.9027663 , 0.93125198, 0.        , 0.        ,
        0.70517255, 0.69994714, 0.35650408, 0.77297609, 0.        ,
        0.45091008, 0.        , 0.41992358, 0.49064529, 0.        ],
       [0.12751407, 0.98462372, 0.93972941, 0.24679409, 0.42325213,
        0.38040088, 0.23166431, 0.55917043, 0.5389853 , 0.38548964,
        0.        , 0.44024211, 0.68240372, 0.        , 0.7506792 ,
        0.76092841, 0.40784591, 0.11569185, 0.63878867, 0.2317411 ],
       [0.35826829, 0.33825579, 0.88331743, 0.83822976, 0.29924224,
        0.        , 0.22805642, 0.80512085, 0.71746001, 0.48851301,
        0.95814413, 0.89374105, 0.25481214, 0

## Ex 1.4 

Generate two sparse Matrix and sum them together, choosing the most appropriate internal representation (i.e. `LIL`, `CSR`, `DOK`...).

#### Hint: Oh c'mon.. :)

In [14]:
# CSR is the most appropriate format here since it supports efficient
# arithmetic operations like addition between sparse matrices
A_sparse = sparse.random(5, 5, density=0.3, format='csr')
B_sparse = sparse.random(5, 5, density=0.3, format='csr')

sum_sparse = A_sparse + B_sparse
print(sum_sparse)
print(sum_sparse.toarray())

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 14 stored elements and shape (5, 5)>
  Coords	Values
  (0, 0)	0.8144706839835401
  (0, 1)	0.217536230911054
  (0, 2)	0.4465324417011688
  (0, 3)	0.7976251990408165
  (0, 4)	0.031179360641079623
  (1, 2)	0.6488798599191623
  (1, 3)	0.11411715695562874
  (1, 4)	0.536853832853208
  (2, 0)	0.020361825035894454
  (2, 1)	0.7534145181367421
  (2, 3)	1.5705610472488887
  (3, 4)	0.4078311860252841
  (4, 0)	0.004920872965476053
  (4, 1)	0.20624093829575196
[[0.81447068 0.21753623 0.44653244 0.7976252  0.03117936]
 [0.         0.         0.64887986 0.11411716 0.53685383]
 [0.02036183 0.75341452 0.         1.57056105 0.        ]
 [0.         0.         0.         0.         0.40783119]
 [0.00492087 0.20624094 0.         0.         0.        ]]
